In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import pandas_ta as ta
from sklearn.model_selection import train_test_split, TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score
from xgboost import XGBClassifier
import shap
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

pd.set_option("display.max_columns", None)

In [ ]:
symbol = "GOOGL"
start  = "2010-01-01"
end    = None

df = yf.download(symbol, start=start, end=end)

# flatten multi-level columns (this is for the recent yfinance versions)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df.head()

In [ ]:
def _merge(df_main, ta_obj):
    """Add TA output to df_main, regardless of Series/DataFrame type."""
    if isinstance(ta_obj, pd.DataFrame):
        return pd.concat([df_main, ta_obj], axis=1)
    else:
        return pd.concat([df_main, ta_obj.rename(ta_obj.name)], axis=1)

def add_indicators(data: pd.DataFrame) -> pd.DataFrame:
    df = data.copy()

    # ── Momentum ──────────────────────────────────────────────
    for length in [5, 10, 15]:
        df[f"rsi_{length}"] = ta.rsi(df["Close"], length=length)
    df["roc_10"] = ta.roc(df["Close"], length=10)
    df["mom_10"] = ta.mom(df["Close"], length=10)

    # ── Oscillators ───────────────────────────────────────────
    df = _merge(df, ta.stochrsi(df["Close"]))            
    df["cci_20"] = ta.cci(df["High"], df["Low"], df["Close"], length=20)
    df["wr_14"]  = ta.willr(df["High"], df["Low"], df["Close"], length=14)
    df = _merge(df, ta.kst(df["Close"]))                 
    df["macd"]   = ta.macd(df["Close"])["MACD_12_26_9"]  

    # ── Trend ────────────────────────────────────────────────
    for length in [5, 10, 20]:
        df[f"sma_{length}"] = ta.sma(df["Close"], length=length)
        df[f"ema_{length}"] = ta.ema(df["Close"], length=length)
    df["vwma_20"] = ta.vwma(df["Close"], df["Volume"], length=20)

    # ── Volatility ───────────────────────────────────────────
    df = _merge(df, ta.bbands(df["Close"], length=20))  
    df["atr_14"] = ta.atr(df["High"], df["Low"], df["Close"], length=14)
    df = _merge(df, ta.kc(df["High"], df["Low"], df["Close"], length=20))

    # ── Volume ───────────────────────────────────────────────
    df["obv"] = ta.obv(df["Close"], df["Volume"])
    df["ad"]  = ta.ad(df["High"], df["Low"], df["Close"], df["Volume"])
    df["efi"] = ta.efi(df["Close"], df["Volume"])
    df["nvi"] = ta.nvi(df["Close"], df["Volume"])
    df["pvi"] = ta.pvi(df["Close"], df["Volume"])

    return df


df_ta = add_indicators(df)
df_ta.tail()


In [ ]:
def generate_label(
        data: pd.DataFrame,
        lookahead: int = 5,
        thresh: float = 0.01,
        col: str = "Close"
) -> pd.Series:
    """
    Label each row from the mean of the *next* `lookahead` closes:
      2 : future_mean ≥ current * (1 + thresh)
      1 : future_mean ≤ current * (1 - thresh)
      0 : otherwise
    """
    future_mean = (
        data[col]
        .shift(-lookahead)
        .rolling(window=lookahead, min_periods=lookahead)
        .mean()
    )
    #data["means"] = future_mean
    pct_change = (future_mean - data[col]) / data[col]

    labels = np.select(
        [pct_change >= thresh, pct_change <= -thresh],
        [2, 1],
        default=0
    )

    return pd.Series(labels, index=data.index)



# Now compute the labels for different lookaheads and thresholds
# --------------------------------------------------------------
lookaheads = [2,4,6,8,10]
thresholds = [0.01, 0.02]

for la in lookaheads:
    for th in thresholds:
        df_ta[f"label_la{la}_th{th:.3f}"] = generate_label(df_ta, lookahead=la, thresh=th)
#---------------------------------------------------------------

df_ta.dropna(inplace=True)
df_ta.head()

In [ ]:
split_idx = int(len(df_ta) * 0.6)
split_idx_val = int(len(df_ta) * 0.8)
train_df, test_df = df_ta.iloc[:split_idx], df_ta.iloc[split_idx:split_idx_val]
val_df = df_ta.iloc[split_idx_val:]

feature_cols = [c for c in df_ta.columns if c not in ["Open","High","Low","Close","Adj Close","Volume"] and not c.startswith("label_")]
print("Total features:", len(feature_cols))

In [ ]:
results = []

for label_col in [c for c in df_ta.columns if c.startswith("label_")]:
    X_train, y_train = train_df[feature_cols], train_df[label_col]
    X_test,  y_test  = test_df[feature_cols],  test_df[label_col]
    
    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.8,
        objective='multi:softprob',
        num_class=3,
        n_jobs=-1,
        eval_metric='mlogloss',
        seed=42
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    acc  = accuracy_score(y_test, preds)
    f1   = f1_score(y_test, preds, average='macro')
    
    results.append({
        "label": label_col,
        "accuracy": acc,
        "f1_macro": f1
    })

pd.DataFrame(results).sort_values("accuracy", ascending=False).head(10)

In [ ]:
best_label = max(results, key=lambda x: x["f1_macro"])["label"]
# print("Best label:", best_label)

#best_label = "label_la6_th0.030"  # Replace with custom label from results

X_train, y_train = train_df[feature_cols], train_df[best_label]
X_test,  y_test  = test_df[feature_cols],  test_df[best_label]

param_grid = {
    "n_estimators": [200, 400, 600],
    "max_depth":   [4, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample":  [0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0]
}
tscv = TimeSeriesSplit(n_splits=5)

xgb = XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    n_jobs=-1,
    eval_metric='mlogloss',
    seed=42
)

grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=tscv,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
)

grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print("Best CV accuracy:", grid.best_score_)

best_model = grid.best_estimator_
test_preds = best_model.predict(X_test)
print(classification_report(y_test, test_preds))

In [ ]:
X_val,  y_val  = val_df[feature_cols],  val_df[best_label]
val_preds = best_model.predict(X_val)
print(classification_report(y_val, val_preds))

In [ ]:
THRESH_1 = 0.55   # class 1  (bearish sell)
THRESH_2 = 0.55  # class 2  (bullish buy)

import numpy as np

def custom_select(proba: np.ndarray,
                  thr1: float = 0.65,
                  thr2: float = 0.65,
                  neutral_label: int = 0) -> np.ndarray:
    """
    Parameters
    ----------
    proba : ndarray, shape (n_samples, 3)
        Class-probability matrix from XGBClassifier.predict_proba().
        Column indices: 0 → neutral / 'no trade', 1 → class 1, 2 → class 2.
    thr1, thr2 : float
        Minimum probability required to accept class 1 or class 2 respectively.
    neutral_label : int
        Label to assign when the confidence criteria are not met.

    Returns
    -------
    pred : ndarray, shape (n_samples,)
        Final class predictions after thresholding.
    """
    # start with everyone assigned to 'neutral'
    pred = np.full(len(proba), neutral_label, dtype=int)

    # --- choose class 1 if it is:
    #     • the highest-probability class AND
    #     • above its own threshold
    mask1 = (proba[:, 1] >= thr1) & (proba[:, 1] > proba[:, 2])
    pred[mask1] = 1

    # --- choose class 2 under analogous conditions
    mask2 = (proba[:, 2] >= thr2) & (proba[:, 2] > proba[:, 1])
    pred[mask2] = 2

    # rest of rows remain neutral_label
    return pred

In [ ]:
import numpy as np

def selective_predict_proba(proba: np.ndarray,
                            thr1: float = 0.65,
                            thr2: float = 0.65) -> np.ndarray:
    """
    Convert a [n_samples, 3] probability matrix into
    class labels 0/1/2 with optional abstention (e.g. -1).
    """
    pred = np.zeros(len(proba), dtype=int)  # default = class 0

    # take class 1 only if it's clearly on top *and* above its threshold
    mask1 = (proba[:, 1] >= thr1) & (proba[:, 1] > proba[:, 2])
    pred[mask1] = 1

    # take class 2 under the analogous rule
    mask2 = (proba[:, 2] >= thr2) & (proba[:, 2] > proba[:, 1])
    pred[mask2] = 2

    # optional: flag low-confidence points instead of forcing them to 0
    # pred[~(mask1 | mask2)] = -1
    return pred

In [ ]:
X_train, y_train = train_df[feature_cols], train_df[best_label]
X_test,  y_test  = test_df[feature_cols],  test_df[best_label]

y_proba = best_model.predict_proba(X_test)
sel_preds = selective_predict_proba(y_proba, THRESH_1, THRESH_2)

test_df["sel_preds"] = sel_preds
print(classification_report(y_test, sel_preds))

y_proba = best_model.predict_proba(X_val)
sel_preds = selective_predict_proba(y_proba, THRESH_1, THRESH_2)
val_df["sel_preds"] = sel_preds

In [ ]:
import plotly.graph_objs as go

def plot_signals(test_df, pred_column='predictions', start=0, length=100):
    """
    Plot OHLC candles from test_df with:
    • purple dots for model signals
    • EMA line
    """
    # Subset
    df_slice = test_df.iloc[start:start+length].copy()
    df_slice.reset_index(inplace=True)  # Reset for plotting

    # Candle plot
    fig = go.Figure(data=[go.Candlestick(
        x=df_slice['index'] if 'index' in df_slice else df_slice.index,
        open=df_slice['Open'],
        high=df_slice['High'],
        low=df_slice['Low'],
        close=df_slice['Close'],
        name='Candles'
    )])

    
    if 'ema' in df_slice.columns:                     # adjust column name if needed
        fig.add_trace(go.Scatter(
            x=df_slice['index'] if 'index' in df_slice else df_slice.index,
            y=df_slice['ema'],
            mode='lines',
            line=dict(width=1.5, dash='dot', color='orange'),
            name='EMA'
        ))
    

    # Bullish signals
    bull_idx = df_slice[df_slice[pred_column] == 2].index
    fig.add_trace(go.Scatter(
        x=df_slice.loc[bull_idx, 'index'] if 'index' in df_slice else bull_idx,
        y=df_slice.loc[bull_idx, 'Low'] - (df_slice['High']-df_slice['Low']).mean()*0.1,
        mode='markers',
        marker=dict(size=7, color='purple', symbol='circle'),
        name='Bullish signal (2)'
    ))

    # Bearish signals
    bear_idx = df_slice[df_slice[pred_column] == 1].index
    fig.add_trace(go.Scatter(
        x=df_slice.loc[bear_idx, 'index'] if 'index' in df_slice else bear_idx,
        y=df_slice.loc[bear_idx, 'High'] + (df_slice['High']-df_slice['Low']).mean()*0.1,
        mode='markers',
        marker=dict(size=7, color='purple', symbol='circle'),
        name='Bearish signal (1)'
    ))


    # Bullish labels
    bull_idx = df_slice[df_slice[best_label] == 2].index
    fig.add_trace(go.Scatter(
        x=df_slice.loc[bull_idx, 'index'] if 'index' in df_slice else bull_idx,
        y=df_slice.loc[bull_idx, 'Low'] - (df_slice['High']-df_slice['Low']).mean()*0.5,
        mode='markers',
        marker=dict(size=7, color='red', symbol='triangle-up'),
        name='Bullish Label (2)'
    ))

    # Bearish labels
    bear_idx = df_slice[df_slice[best_label] == 1].index
    fig.add_trace(go.Scatter(
        x=df_slice.loc[bear_idx, 'index'] if 'index' in df_slice else bear_idx,
        y=df_slice.loc[bear_idx, 'High'] + (df_slice['High']-df_slice['Low']).mean()*0.5,
        mode='markers',
        marker=dict(size=7, color='red', symbol='triangle-down'),
        name='Bearish Label (1)'
    ))

    fig.update_layout(
        title=f"Signals (purple) & EMA from index {start} to {start+length}",
        xaxis_title="Index",
        yaxis_title="Price",
        showlegend=True,
        height=600,
        width=1000
    )
    fig.show()


plot_signals(val_df, pred_column="sel_preds", start=0, length=160)

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, matthews_corrcoef,
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, auc
)
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

y_pred   = best_model.predict(X_test)
y_proba  = best_model.predict_proba(X_test)

print("----- Classification report -----")
print(classification_report(y_test, y_pred, digits=4))

summary = {
    "Accuracy"          : accuracy_score(y_test, y_pred),
    "Balanced accuracy" : balanced_accuracy_score(y_test, y_pred),
    "Macro-F1"          : f1_score(y_test, y_pred, average="macro"),
    "Macro-Precision"   : precision_score(y_test, y_pred, average="macro"),
    "Macro-Recall"      : recall_score(y_test, y_pred, average="macro"),
    "MCC"               : matthews_corrcoef(y_test, y_pred)
}
pd.DataFrame(summary, index=["Score"]).T.style.format("{:.4f}")

In [ ]:
# Confusion-matrix heat-map (normalised row-wise)
cm  = confusion_matrix(y_test, y_pred, labels=[0,1,2])
cmn = cm / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(5,4))
sns.heatmap(cmn, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=["0","1","2"], yticklabels=["0","1","2"])
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.title("Normalised confusion matrix")
plt.tight_layout()
plt.show()

In [ ]:
# ROC curves (one-vs-rest) + macro/micro averages
n_classes = 3
fpr = dict(); tpr = dict(); roc_auc = dict()

# one-vs-rest curves
for c in range(n_classes):
    fpr[c], tpr[c], _ = roc_curve((y_test == c).astype(int), y_proba[:, c])
    roc_auc[c] = auc(fpr[c], tpr[c])

# micro-average
fpr["micro"], tpr["micro"], _ = roc_curve(
    pd.get_dummies(y_test).values.ravel(), y_proba.ravel()
)
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

# macro-average (average the FPR points first, then AU-C)
all_fpr = np.unique(np.concatenate([fpr[c] for c in range(n_classes)]))
mean_tpr = np.zeros_like(all_fpr)
for c in range(n_classes):
    mean_tpr += np.interp(all_fpr, fpr[c], tpr[c])
mean_tpr /= n_classes
fpr["macro"], tpr["macro"] = all_fpr, mean_tpr
roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

# ---------- Plot ----------
plt.figure(figsize=(7,5))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]  # default matplotlib palette

for c, col in zip(range(n_classes), colors):
    plt.plot(fpr[c], tpr[c], color=col,
             label=f"Class {c} (AUC = {roc_auc[c]:.3f})", lw=1.5)

# macro & micro
plt.plot(fpr["micro"], tpr["micro"], linestyle="--", color="black",
         label=f"micro-avg (AUC = {roc_auc['micro']:.3f})", lw=1.5)
plt.plot(fpr["macro"], tpr["macro"], linestyle=":", color="dimgray",
         label=f"macro-avg (AUC = {roc_auc['macro']:.3f})", lw=2)

plt.plot([0,1], [0,1], "k--", lw=0.6)
plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC curves – one-vs-rest")
plt.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
print("\nArea-under-curve (AUC) scores:")
auc_tbl = pd.DataFrame(
    {f"Class {c}": roc_auc[c] for c in range(n_classes)} |
    {"micro-avg": roc_auc["micro"], "macro-avg": roc_auc["macro"]},
    index=["AUC"]
).T
auc_tbl.style.format("{:.4f}")

In [ ]:
# ==================================================
# Walk-Forward (Rolling window) Validation Approach
# ==================================================

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
import pandas as pd

n_splits = 5  # change this for more/less rolling windows
tscv = TimeSeriesSplit(n_splits=n_splits)

fold_metrics = []

print(f"\nWalk-forward validation ({n_splits} splits) on label: {best_label}\n")
for fold, (train_idx, test_idx) in enumerate(tscv.split(df_ta)):
    train = df_ta.iloc[train_idx]
    test = df_ta.iloc[test_idx]
    X_train, y_train = train[feature_cols], train[best_label]
    X_test, y_test = test[feature_cols], test[best_label]

    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.8,
        objective='multi:softprob',
        num_class=3,
        n_jobs=-1,
        eval_metric='mlogloss',
        seed=42,
        use_label_encoder=False
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    metrics = {
        "fold": fold+1,
        "start": test.index[0],
        "end": test.index[-1],
        "accuracy": accuracy_score(y_test, y_pred),
        "f1_macro": f1_score(y_test, y_pred, average="macro"),
        "precision_macro": precision_score(y_test, y_pred, average="macro"),
        "recall_macro": recall_score(y_test, y_pred, average="macro"),
    }
    fold_metrics.append(metrics)
    print(f"Fold {fold+1}: {test.index[0]} to {test.index[-1]}")
    print(f"  Accuracy: {metrics['accuracy']:.4f}, F1-macro: {metrics['f1_macro']:.4f}\n")

# summary DataFrame
results_df = pd.DataFrame(fold_metrics)
display(results_df)

print("\nMean (±std) metrics across folds:")
for m in ["accuracy", "f1_macro", "precision_macro", "recall_macro"]:
    print(f"{m:15}: {results_df[m].mean():.4f} ± {results_df[m].std():.4f}")